In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

import database

# Main

In [2]:
API_KEY = os.environ['KEY']
SERIES_IDS = ['UNRATE', 'PAYEMS', 'ICSA', 'INDPRO', 'JTSJOL']

## Get data for db table `FREDSeries`
- SeriesID
- Title 
- DefaultFrequency
- Units
- ObservationStart
- FRED_LastUpdateLocalTime
- FRED_LastUpdateUTC (must be converted from CD/CDT to UTC)
- CreationDate `TEXT NOT NULL DEFAULT (datetime('now'))`

<b style="color:#fa0240;">Handle missing entries fetched from FRED</b>

In [25]:
fred_time_stamp = "2026-08-07 08:31:41-05"
fred_time = datetime.strptime(fred_time_stamp + "00", "%Y-%m-%d %H:%M:%S%z")
utc_time = fred_time.astimezone(timezone.utc)

print(fred_time)
print(utc_time)

2026-08-07 08:31:41-05:00
2026-08-07 13:31:41+00:00


In [ ]:
fred_series_params = {
    "api_key": API_KEY,
    "file_type": "json"
}

def requester_fred_series(
        url="https://api.stlouisfed.org/fred/series",
        base_params=fred_series_params,
        SeriesIDs=SERIES_IDS
    ):
    results = []

    with requests.Session() as session:
        for series_id in SERIES_IDS:
            params = {
                **base_params,
                "series_id": series_id
            }

            r = session.get(url=url, params=params, timeout=30)
            r.raise_for_status()

            data = r.json()

            fred_local_time = datetime.strptime(data['seriess'][0]['last_updated'] + "00", "%Y-%m-%d %H:%M:%S%z")
            fred_utc_time = fred_local_time.astimezone(timezone.utc)

            fred_series_data = {
                "series_id": data['seriess'][0]['id'],
                "title": data['seriess'][0]['title'],
                "frequency": data['seriess'][0]['frequency'],
                "units": data['seriess'][0]['units'],
                "observation_start": data['seriess'][0]['observation_start'],
                "last_update_local_time": fred_local_time.strftime("%Y-%m-%d %H:%M:%S"),
                "last_update_utc":fred_utc_time.strftime("%Y-%m-%d %H:%M:%S"),
            }

            results.append(fred_series_data)

    return results


requester_fred_series()

## Get data for db table `FREDObservations`

- SeriesID
- ObservationDate
- NativeValue
- DateFetched `TEXT NOT NULL DEFAULT (datetime('now')),`

<s><b style="color:#fa0240;">Use for `observation_start` date the earliest common date among all data series</b></s>

<b style="color:DodgerBlue;">Just redownload everything every month since it is not so much</b>

In [30]:
earliest_common_observation_start = "2000-12-01"

def requester_fred_observations():
    url = 'https://api.stlouisfed.org/fred/series/observations'
    results = []

    with requests.Session() as session:
        for series_id in SERIES_IDS:
            params = {
                "api_key": API_KEY,
                "file_type": "json",
                "sort_order": "asc",
                "units": "lin",
                "observation_start": earliest_common_observation_start,
                "series_id": series_id
            }

            r = session.get(url=url, params=params, timeout=30)
            r.raise_for_status()

            data = r.json()
            entries = data['observations']

            for entry in entries:
                fred_observations_data = {
                    "series_id": series_id,
                    "observation_date": entry['date'],
                    "native_value": entry['value']
                }
            
                results.append(fred_observations_data)

    return results

## Prepare data from `FREDObservations` for db table `MonthlyFeatures`
- SeriesID
- MonthYear
- MonthlyValue
- DateCalculated `TEXT NOT NULL DEFAULT (datetime('now')),`

<b style="color:#fa0240;">Implement a checker if the dataset for the month is complete to kick of the ML pipeline</b>

In [179]:
native_observations = database.db_requester_fetchall("all_obs_FREDObservations.sql")

In [180]:
# Convert all dates to YYYY-MM and timestamp is dropped
arr = np.array(native_observations)
arr = np.delete(arr,[False, False, False, True], 1)
arr[:][:,1] = arr[:][:,1].astype("datetime64[M]")
# arr
idx_icsa = np.where(arr=='ICSA')[0]
none_icsa_arr = np.delete(arr, idx_icsa, 0)
none_icsa_arr

array([['UNRATE', datetime.date(2000, 12, 1), 3.9],
       ['UNRATE', datetime.date(2001, 1, 1), 4.2],
       ['UNRATE', datetime.date(2001, 2, 1), 4.2],
       ...,
       ['JTSJOL', datetime.date(2026, 4, 1), 7585.0],
       ['JTSJOL', datetime.date(2026, 5, 1), 7537.0],
       ['JTSJOL', datetime.date(2026, 6, 1), 7359.0]],
      shape=(1230, 3), dtype=object)

In [181]:
icsa_arr = arr[idx_icsa]

df_icsa = pd.DataFrame(icsa_arr)
column_names = ['series_id', 'month_year', 'monthly_value']
df_icsa.columns = column_names

df_monthly_icsa = (
    df_icsa.groupby("month_year", as_index=False).agg({
        "series_id": "last",
        "monthly_value": lambda x: None if x.isna().any() else x.sum()
    })
)
df_monthly_icsa.insert(
    0,
    df_monthly_icsa.columns[1],
    df_monthly_icsa.pop(df_monthly_icsa.columns[1])
)

df_monthly_icsa

,series_id,month_year,monthly_value
0,ICSA,2000-12-01,1730000.0
1,ICSA,2001-01-01,1360000.0
2,ICSA,2001-02-01,1485000.0
3,ICSA,2001-03-01,1936000.0
4,ICSA,2001-04-01,1587000.0
...,...,...,...
304,ICSA,2026-04-01,831000.0
305,ICSA,2026-05-01,1058000.0
306,ICSA,2026-06-01,890000.0
307,ICSA,2026-07-01,813000.0


In [182]:
df_none_ics = pd.DataFrame(none_icsa_arr)
df_none_ics.columns = column_names
df_none_ics

,series_id,month_year,monthly_value
0,UNRATE,2000-12-01,3.9
1,UNRATE,2001-01-01,4.2
2,UNRATE,2001-02-01,4.2
3,UNRATE,2001-03-01,4.3
4,UNRATE,2001-04-01,4.4
...,...,...,...
1225,JTSJOL,2026-02-01,6922.0
1226,JTSJOL,2026-03-01,6887.0
1227,JTSJOL,2026-04-01,7585.0
1228,JTSJOL,2026-05-01,7537.0


In [183]:
df_all_monthly = pd.concat([df_none_ics, df_monthly_icsa],ignore_index=True)
df_all_monthly = df_all_monthly.astype(object).where(
    pd.notna(df_all_monthly),
    None
)
db_ready_monthly_values = df_all_monthly.to_dict(orient='records')
df_all_monthly

,series_id,month_year,monthly_value
0,UNRATE,2000-12-01,3.9
1,UNRATE,2001-01-01,4.2
2,UNRATE,2001-02-01,4.2
3,UNRATE,2001-03-01,4.3
4,UNRATE,2001-04-01,4.4
...,...,...,...
1534,ICSA,2026-04-01,831000.0
1535,ICSA,2026-05-01,1058000.0
1536,ICSA,2026-06-01,890000.0
1537,ICSA,2026-07-01,813000.0


In [184]:
df_all_monthly[df_all_monthly['monthly_value'].isnull()]

,series_id,month_year,monthly_value
298,UNRATE,2025-10-01,None


In [172]:
db_ready_monthly_values[0:2]

[{'series_id': 'UNRATE',
  'month_year': datetime.date(2000, 12, 1),
  'monthly_value': 3.9},
 {'series_id': 'UNRATE',
  'month_year': datetime.date(2001, 1, 1),
  'monthly_value': 4.2}]

# Old

In [ ]:
base_url = 'https://api.stlouisfed.org/fred/series/observations'

base_params = {
    'api_key': API_KEY,
    'file_type': 'json',
    'sort_order': 'asc',
    "units": "lin"
}

def requester(series_id, observation_start, observation_end='9999-12-31', url=base_url):
    params = base_params.copy()
    params.update({
        'series_id': series_id,
        'observation_start': observation_start,
        'observation_end': observation_end
    })

    r = requests.get(url=url, params=params)

    if r.status_code == 200:
        return r.json()
    else:
        status = f"{params['series_id']} raised an error code {r.status_code}"
        return status

In [104]:
base_params

{'api_key': '937b94a2d841ad9bfad6ec242a4ded45',
 'file_type': 'json',
 'sort_order': 'asc',
 'units': 'lin'}

In [105]:
response = requester('ICSA', '2000-01-01')

In [118]:
response.keys()

dict_keys(['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations'])

In [110]:
df = pd.DataFrame(response['observations'])
df

,realtime_start,realtime_end,date,value
0,2026-08-04,2026-08-04,2000-01-01,286000
1,2026-08-04,2026-08-04,2000-01-08,298000
2,2026-08-04,2026-08-04,2000-01-15,289000
3,2026-08-04,2026-08-04,2000-01-22,284000
4,2026-08-04,2026-08-04,2000-01-29,285000
...,...,...,...,...
1382,2026-08-04,2026-08-04,2026-06-27,217000
1383,2026-08-04,2026-08-04,2026-07-04,217000
1384,2026-08-04,2026-08-04,2026-07-11,209000
1385,2026-08-04,2026-08-04,2026-07-18,188000


# Testing

In [ ]:
payload = {
    'series_id': 'ICSA',
    'api_key': API_KEY,
    'file_type': 'json'
}

r = requests.get('https://api.stlouisfed.org/fred/series', params=payload)

In [46]:
print(r.json())
print(f'observations_start: {r.json()['seriess'][0]['id']}')
print(f'observations_start: {r.json()['seriess'][0]['observation_start']}')
print(f'observations_end: {r.json()['seriess'][0]['observation_end']}')
print(f'frequency: {r.json()['seriess'][0]['frequency']}')
print(f'frequency_short: {r.json()['seriess'][0]['frequency_short']}')
print(f'units: {r.json()['seriess'][0]['units']}')



{'realtime_start': '2026-07-30', 'realtime_end': '2026-07-30', 'seriess': [{'id': 'ICSA', 'realtime_start': '2026-07-30', 'realtime_end': '2026-07-30', 'title': 'Initial Claims', 'observation_start': '1967-01-07', 'observation_end': '2026-07-25', 'frequency': 'Weekly, Ending Saturday', 'frequency_short': 'W', 'units': 'Number', 'units_short': 'Number', 'seasonal_adjustment': 'Seasonally Adjusted', 'seasonal_adjustment_short': 'SA', 'last_updated': '2026-07-30 07:36:19-05', 'popularity': 81, 'notes': 'An initial claim is a claim filed by an unemployed individual after a separation from an employer. The claim requests a determination of basic eligibility for the Unemployment Insurance program.'}]}
observations_start: ICSA
observations_start: 1967-01-07
observations_end: 2026-07-25
frequency: Weekly, Ending Saturday
frequency_short: W
units: Number


In [89]:
r.json()

{'realtime_start': '2026-07-30',
 'realtime_end': '2026-07-30',
 'seriess': [{'id': 'ICSA',
   'realtime_start': '2026-07-30',
   'realtime_end': '2026-07-30',
   'title': 'Initial Claims',
   'observation_start': '1967-01-07',
   'observation_end': '2026-07-25',
   'frequency': 'Weekly, Ending Saturday',
   'frequency_short': 'W',
   'units': 'Number',
   'units_short': 'Number',
   'seasonal_adjustment': 'Seasonally Adjusted',
   'seasonal_adjustment_short': 'SA',
   'last_updated': '2026-07-30 07:36:19-05',
   'popularity': 81,
   'notes': 'An initial claim is a claim filed by an unemployed individual after a separation from an employer. The claim requests a determination of basic eligibility for the Unemployment Insurance program.'}]}

In [111]:
payload.update({'frequency': 'm'})
payload.update({'sort_order': 'desc'})
r_obsv = requests.get(base_url, params=payload)

In [112]:
print(r_obsv.status_code)

200


In [ ]:
arr = np.array(native_observations)
icsa_arr = arr[np.where(arr=='ICSA')[0]]

dates = icsa_arr[:][:,1].astype("datetime64[M]")

icsa_arr[:][:,1] = dates

unique_months = np.unique(dates)
values = icsa_arr[:,2]

monthly_icsa_arr = np.array([
    values[dates == month].sum()
    for month in unique_months
])

monthly_icsa_arr

# dates = arr[idx[0]][:18][:,1].astype("datetime64[D]")

